# Ejercicio 2: comparar con evidencia y elegir

El mismo `viajes_features.csv` y la misma preparación importada. Mismo esqueleto del Ejercicio 1, con los cinco modelos de la Semana 10 en un experimento propio. Al final eliges el mejor desde el registro, no desde tu memoria, y compruebas que el modelo guardado reproduce su número.

Los pasos 1 a 3 tienen `TODO`; los pasos 4 y 5 solo dicen qué variable dejar. La última celda imprime una comprobación por paso; no la modifiques. Corre todas las celdas una vez antes de empezar (**Run All**). Si vuelves a correr las celdas que entrenan, aparecen corridas repetidas en el registro: es lo esperado.

In [1]:
import hashlib
import logging
import time

import mlflow
import mlflow.sklearn
import pandas as pd
from sklearn.dummy import DummyRegressor
from sklearn.ensemble import GradientBoostingRegressor, RandomForestRegressor
from sklearn.linear_model import Ridge
from sklearn.metrics import mean_absolute_error
from sklearn.neural_network import MLPRegressor
from sklearn.pipeline import Pipeline
from preparacion_taxis import dividir, preparacion

# MLflow avisa en cada corrida que el modelo se guarda sin firma (la
# descripcion de sus columnas de entrada). La firma importa al servir el
# modelo, en la Semana 12; hoy solo ensucia la salida.
logging.getLogger("mlflow").setLevel(logging.ERROR)

viajes = pd.read_csv("viajes_features.csv")
X_ent, X_prueba, y_ent, y_prueba = dividir(viajes)
print("Entrenamiento:", X_ent.shape, "Prueba:", X_prueba.shape)


def huella(ruta):
    with open(ruta, "rb") as archivo:
        return hashlib.md5(archivo.read()).hexdigest()[:12]


huella_datos = huella("viajes_features.csv")
EXPERIMENTO = "tarifa-taxis-comparacion"

Entrenamiento: (2380, 10) Prueba: (595, 10)


## Paso 1: el experimento

In [2]:
mlflow.set_experiment(EXPERIMENTO)

<Experiment: artifact_location='mlflow-artifacts:/2', creation_time=1793824218322, experiment_id='2', last_update_time=1793824218322, lifecycle_stage='active', name='tarifa-taxis-comparacion', tags={}>

## Paso 2: entrenar y registrar

In [3]:
def evaluar(nombre, modelo, X_ent=X_ent, X_prueba=X_prueba, y_ent=y_ent, y_prueba=y_prueba, semilla_division=42):
    pipeline = Pipeline([("preparar", preparacion()), ("modelo", modelo)])
    with mlflow.start_run(run_name=nombre):
        mlflow.log_params({**modelo.get_params(), "semilla_division": semilla_division,
                           "huella_datos": huella_datos, "filas_entrenamiento": len(X_ent)})
        inicio = time.perf_counter()
        pipeline.fit(X_ent, y_ent)
        segundos = time.perf_counter() - inicio
        mae_ent = mean_absolute_error(y_ent, pipeline.predict(X_ent))
        mae_prueba = mean_absolute_error(y_prueba, pipeline.predict(X_prueba))
        mlflow.log_metrics({"mae_entrenamiento": mae_ent, "mae_prueba": mae_prueba, "segundos": segundos})
        mlflow.sklearn.log_model(pipeline, "modelo")
    print(f"{nombre}: entrenamiento {mae_ent:.6f}; prueba {mae_prueba:.6f}")


## Paso 3: los cinco modelos de la Semana 10

In [4]:
modelos = [
    ("linea base", DummyRegressor(strategy="median")),
    ("ridge", Ridge(alpha=10.0)),
    ("bosque aleatorio", RandomForestRegressor(n_estimators=200, random_state=42)),
    ("gradient boosting", GradientBoostingRegressor(random_state=42)),
    ("red neuronal", MLPRegressor(hidden_layer_sizes=(64,), max_iter=1000, random_state=42)),
]
for nombre, modelo in modelos:
    evaluar(nombre, modelo)


linea base: entrenamiento 13.199786; prueba 12.530824


ridge: entrenamiento 2.493893; prueba 2.556469


bosque aleatorio: entrenamiento 0.473949; prueba 1.301589


gradient boosting: entrenamiento 1.070539; prueba 1.275352


red neuronal: entrenamiento 1.088204; prueba 1.169505


## Paso 4: el registro, ordenado

In [5]:
experimento = mlflow.get_experiment_by_name(EXPERIMENTO)
corridas = mlflow.search_runs(experiment_ids=[experimento.experiment_id], order_by=["metrics.mae_prueba ASC"]).reset_index(drop=True)
mejor_run_id = corridas.loc[0, "run_id"]
print(corridas[["run_id", "tags.mlflow.runName", "metrics.mae_prueba", "metrics.segundos"]].to_string(index=False))


                          run_id tags.mlflow.runName  metrics.mae_prueba  metrics.segundos
f11afdb381b6447ba15c90cb9206412f        red neuronal            1.169505          0.939561
f92537bdfb984ed9923d4ec4f2803f21   gradient boosting            1.275352          0.127256
9404e6191f14419d9dc929bfcac63f29    bosque aleatorio            1.301589          0.721028
b3d3a91b3de64e3bbf8c95085fb2f1fd               ridge            2.556469          0.012327
b7dc055dc53b4336af75fc447dd74564          linea base           12.530824          0.006563


## Paso 5: marcar el ganador y reproducirlo

In [6]:
mlflow.MlflowClient().set_tag(mejor_run_id, "candidato", "final")
modelo_final = mlflow.sklearn.load_model("runs:/" + mejor_run_id + "/modelo")
mae_reproducido = mean_absolute_error(y_prueba, modelo_final.predict(X_prueba))
print("MAE reproducido:", mae_reproducido)


MAE reproducido: 1.1695052296355013


## Comprobación

In [7]:
def comprobar(numero, condicion):
    if condicion:
        print(f"Paso {numero}: OK")
    else:
        print(f"Paso {numero}: revisa")


experimento = mlflow.get_experiment_by_name(EXPERIMENTO)
registradas = None
if experimento is not None:
    registradas = mlflow.search_runs(experiment_ids=[experimento.experiment_id])
nombres = set()
if registradas is not None and len(registradas) > 0:
    nombres = set(registradas["tags.mlflow.runName"])

comprobar(1, experimento is not None)
comprobar(2, registradas is not None and "params.huella_datos" in registradas and (registradas["params.huella_datos"] == huella_datos).all())
comprobar(3, {"linea base", "ridge", "bosque aleatorio", "gradient boosting", "red neuronal"} <= nombres and registradas["metrics.mae_prueba"].notna().all())
comprobar(4, corridas is not None and mejor_run_id == corridas.loc[0, "run_id"] and corridas.loc[0, "tags.mlflow.runName"] == "red neuronal")
etiqueta = None
if mejor_run_id is not None:
    etiqueta = mlflow.MlflowClient().get_run(mejor_run_id).data.tags.get("candidato")
comprobar(5, etiqueta == "final" and mae_reproducido is not None and abs(mae_reproducido - corridas.loc[0, "metrics.mae_prueba"]) < 1e-9)

Paso 1: OK
Paso 2: OK
Paso 3: OK
Paso 4: OK
Paso 5: OK


## Si terminas antes

Abre la interfaz de MLflow (pestaña **Ports** de VS Code, puerto 5050, ícono del globo), entra al experimento `tarifa-taxis-comparacion`, selecciona las cinco corridas y compáralas con **Compare**. Además del error, ¿qué te dice el registro de la red neuronal que no te diría una tabla escrita a mano?